In [1]:
import scipy as sp
import numpy as np
import math
import pyscf

from gbasis.parsers import parse_nwchem
from gbasis.parsers import parse_gbs, make_contractions
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.overlap_asymm import overlap_integral_asymmetric
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral



elec_basis_dict = parse_nwchem("def2-SVP.nw")
nuc_basis_dict = parse_nwchem("DZSNB.nw")

dist_a = 0.7414
dist_bohr = dist_a * 1.8897259886


mass_proton = 1874.0  #mass of proton in atomic units (electron masses)

# Centers of electron orbitals (just the H atoms position)
elec_atoms = ["H", "H"]
elec_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, dist_bohr]])

# Centers of nuclear orbitals for protons
nuc_atoms = ["Q", "Q"]
nuc_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, dist_bohr]])


# Construct full molecular orbital  basis from atomic orbitals centered at the positions "atcoords"
elec_basis = make_contractions(elec_basis_dict, elec_atoms, elec_atcoords, coord_types="cartesian")
nuc_basis = make_contractions(nuc_basis_dict, nuc_atoms, nuc_atcoords, coord_types="cartesian")
full_basis = elec_basis + nuc_basis # Bases are tuples of contracted gaussian objects. NOTE that the overlap

# symmetric orthogonalization of AOs. Szabo sec 3.4.5
elec_overlap = overlap_integral(elec_basis)
elec_ortho = np.linalg.inv(sp.linalg.sqrtm(elec_overlap))  # Transform needed to get orthogonal MOs

# RHF calculation to get MOs that will be used for the quantum algorihtm
mol = pyscf.M(
    atom = 'H 0 0 0; H 0 0 0.7414',  # in Angstrom
    basis = 'def2-SVP',
    symmetry = True,
)
#=
hf = mol.HF().run()
elec_ortho = hf.mo_coeff.T # probably should get the integrals from pyscf too? in case they dont match gbasis. or figure a way to parse pyscf w gbasis.

nuc_overlap = overlap_integral(nuc_basis)
nuc_ortho = np.linalg.inv(sp.linalg.sqrtm(nuc_overlap))  # Transform needed to get orthogonal MOs

# basis function count
elec_bc = elec_overlap.shape[0]
nuc_bc = nuc_overlap.shape[0]

full_ortho = np.block([[elec_ortho, np.zeros((elec_bc, nuc_bc))],
                       [np.zeros((nuc_bc, elec_bc)), nuc_ortho]])

# Note the "non-zero" overlap between nuclear and electronic orbitals. They actually are orthogonal in the Hilbert space though.
elec_ke = kinetic_energy_integral(elec_basis, transform=elec_ortho)

print(elec_ke)

nuc_ke = kinetic_energy_integral(nuc_basis, transform=nuc_ortho) / mass_proton
print(nuc_ke)

ee_coulomb =electron_repulsion_integral(elec_basis, transform=elec_ortho)

nn_coulomb = electron_repulsion_integral(nuc_basis, transform=nuc_ortho)

coulomb = electron_repulsion_integral(full_basis, transform=full_ortho)

en_coulomb = coulomb[0:elec_bc, elec_bc:elec_bc+nuc_bc, 0:elec_bc, elec_bc:elec_bc+nuc_bc]

converged SCF energy = -1.12890637848863
[[ 5.48325427e-01  1.41436617e-17 -5.23593808e-01  2.36681533e-17
   0.00000000e+00  0.00000000e+00  6.82018154e-02  0.00000000e+00
   0.00000000e+00  4.27449230e-17]
 [ 1.81091258e-17  3.42058998e-01 -2.25802272e-17 -2.78557095e-01
   0.00000000e+00  0.00000000e+00 -5.37575304e-17  0.00000000e+00
   0.00000000e+00  1.37513852e-01]
 [-5.23593808e-01 -2.34330854e-17  8.81740748e-01  6.17350553e-18
   0.00000000e+00  0.00000000e+00 -6.75634763e-02  0.00000000e+00
   0.00000000e+00 -3.96746077e-16]
 [ 6.97212122e-17 -2.78557095e-01 -4.23507104e-17  1.72166758e+00
   0.00000000e+00  0.00000000e+00  3.42671328e-17  0.00000000e+00
   0.00000000e+00 -5.52919296e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   1.80326384e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   5.89586070e-17  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  1.80326384e+00  0.00000000e+00  5.89586